In [ ]:
# 1. Leer el JSON original (está en raw/url/)
raw_json_path = "s3://isabelalake/raw/url/tmdb_metadata.json"
df = spark.read.option("multiline", "true").json(raw_json_path)

# 2. Verificar que se leyó correctamente
print("Número de películas:", df.count())
df.printSchema()
df.show(5)

# 3. Escribir como Parquet en la ubicación que espera Hive
output_path = "s3://isabelalake/trusted/tmdb/"
df.write.mode("overwrite").parquet(output_path)
print(f"✅ Datos guardados en {output_path}")

# 4. Verificar que el Parquet se puede leer
df_check = spark.read.parquet(output_path)
print("Filas en Parquet:", df_check.count())
df_check.show(5)

In [ ]:
# Leer como texto plano para ver el contenido real
gdp_path = "s3://isabelalake/raw/ec2/gdp_data.json"
try:
    text_df = spark.read.text(gdp_path).limit(5)
    text_df.show(truncate=False)
except Exception as e:
    print(f"Error: {e}")

In [ ]:
from pyspark.sql.functions import col, from_json, schema_of_json
from pyspark.sql.types import StructType, StructField, ArrayType, StringType, DoubleType, IntegerType

# Leer como texto
df_text = spark.read.text(raw_gdp_path)

# El contenido completo está en una sola línea (o varias). Vamos a tomar la primera.
json_str = df_text.collect()[0][0]

# Parsear con Python (es pequeño, se puede hacer así)
import json
data = json.loads(json_str)

# data es un diccionario. Lo que nos interesa es el array bajo la clave "_2"? Revisa la estructura impresa:
# En tu salida, el JSON comienza con: {"page":1,"pages":352,... , luego un array. La estructura es:
# [ { "page":..., "pages":..., ... }, [ {...}, {...}, ... ] ]
# O sea, es un array de dos elementos: el primero es metadatos, el segundo es el array de datos.
# En la representación que pegaste, se ve: [{"page":1,"pages":352,...}, [{"indicator":{...}, ...}, ...]]
# Por lo tanto, data es una lista de dos elementos. El segundo elemento es la lista de países/años.

# Extraemos el array de datos (segundo elemento)
array_data = data[1]   # asumiendo que es el segundo elemento

# Convertir a DataFrame de Spark
df_gdp_clean = spark.createDataFrame(array_data)

# Mostrar el esquema y los datos
print("Esquema de datos de GDP per cápita:")
df_gdp_clean.printSchema()
df_gdp_clean.show(5, truncate=False)

# Ahora guardar en trusted
output_gdp_path = "s3://isabelalake/trusted/gdp/"
df_gdp_clean.write.mode("overwrite").parquet(output_gdp_path)
print(f"✅ Datos GDP guardados en {output_gdp_path}")

# Verificar
spark.read.parquet(output_gdp_path).show(5)

In [ ]:
ratings_path = "s3://isabelalake/raw/rds/ratings.csv"
try:
    df_ratings_test = spark.read.option("header", "true").csv(ratings_path)
    print("Columnas:", df_ratings_test.columns)
    df_ratings_test.show(5)
except Exception as e:
    print("Error:", e)
    # Si hay error, mostrar las primeras líneas como texto
    spark.read.text(ratings_path).show(5, truncate=False)

In [ ]:
import json
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

# Leer el archivo JSON como texto
raw_gdp_path = "s3://isabelalake/raw/ec2/gdp_data.json"
text_df = spark.read.text(raw_gdp_path)
json_str = text_df.collect()[0][0]

# Parsear con Python
data = json.loads(json_str)

# La estructura es: [ {metadatos}, [array_de_datos] ]
# El array de datos está en el segundo elemento
gdp_array = data[1]

# Crear DataFrame de Spark
df_gdp = spark.createDataFrame(gdp_array)

# Mostrar esquema y datos
print("Esquema de GDP:")
df_gdp.printSchema()
print("\nMuestra (primeras 5 filas):")
df_gdp.show(5, truncate=False)

# Guardar en trusted/gdp/ como Parquet
output_gdp_path = "s3://isabelalake/trusted/gdp/"
df_gdp.write.mode("overwrite").parquet(output_gdp_path)
print(f"\n✅ Datos GDP guardados en {output_gdp_path}")

# Verificar
print("\nVerificando lectura del Parquet guardado:")
spark.read.parquet(output_gdp_path).show(5, truncate=False)

In [ ]:
# Leer ratings.csv (ya sabes que tiene cabecera y columnas correctas)
ratings_path = "s3://isabelalake/raw/rds/ratings.csv"
df_ratings = spark.read.option("header", "true").csv(ratings_path)

# Convertir rating a float (opcional, pero útil)
from pyspark.sql.functions import col
df_ratings = df_ratings.withColumn("rating", col("rating").cast("float"))

# Guardar como Parquet en trusted/ratings/
output_ratings = "s3://isabelalake/trusted/ratings/"
df_ratings.write.mode("overwrite").parquet(output_ratings)
print(f"✅ Ratings guardados en {output_ratings}")

# Verificar
spark.read.parquet(output_ratings).show(5)